# Fashion-MNIST Convolutional Autoencoder

IMSA coursework publication copy. Saved outputs were cleared; rerun before citing results.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split


In [ ]:
# 1) Load dataset
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

x_train, x_val, y_train, y_val = train_test_split(
    x_train_full,
    y_train_full,
    test_size=0.1,
    random_state=42,
    stratify=y_train_full
)

print('Train shape:', x_train.shape)
print('Validation shape:', x_val.shape)
print('Test shape:', x_test.shape)

# 2) Preprocessing
x_train = x_train.astype('float32') / 255.0
x_val = x_val.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

x_train = np.expand_dims(x_train, axis=-1)
x_val = np.expand_dims(x_val, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

print('After preprocessing:')
print('x_train:', x_train.shape, x_train.dtype, f'min={x_train.min():.3f}, max={x_train.max():.3f}')
print('x_val  :', x_val.shape, x_val.dtype, f'min={x_val.min():.3f}, max={x_val.max():.3f}')
print('x_test :', x_test.shape, x_test.dtype, f'min={x_test.min():.3f}, max={x_test.max():.3f}')

In [ ]:
# 3) Build a Convolutional Autoencoder
latent_dim = 64

# Encoder
encoder_inputs = keras.Input(shape=(28, 28, 1), name='encoder_input')
x = layers.Conv2D(32, 3, activation='relu', padding='same')(encoder_inputs)
x = layers.MaxPooling2D(2, padding='same')(x)      
x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
x = layers.MaxPooling2D(2, padding='same')(x)        
shape_before_flatten = tf.keras.backend.int_shape(x)[1:]
x = layers.Flatten()(x)
latent = layers.Dense(latent_dim, name='latent_vector')(x)

encoder = keras.Model(encoder_inputs, latent, name='encoder')

# Decoder
decoder_inputs = keras.Input(shape=(latent_dim,), name='decoder_input')
x = layers.Dense(np.prod(shape_before_flatten), activation='relu')(decoder_inputs)
x = layers.Reshape(shape_before_flatten)(x)
x = layers.Conv2DTranspose(64, 3, strides=2, activation='relu', padding='same')(x)   
x = layers.Conv2DTranspose(32, 3, strides=2, activation='relu', padding='same')(x)  
decoder_outputs = layers.Conv2D(1, 3, activation='sigmoid', padding='same', name='reconstruction')(x)

decoder = keras.Model(decoder_inputs, decoder_outputs, name='decoder')

# Full Autoencoder
autoencoder_input = keras.Input(shape=(28, 28, 1), name='autoencoder_input')
encoded = encoder(autoencoder_input)
decoded = decoder(encoded)
autoencoder = keras.Model(autoencoder_input, decoded, name='conv_autoencoder')

autoencoder.compile(optimizer='adam', loss='mse')

autoencoder.summary()

In [ ]:
# 4) Train model
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
]

history = autoencoder.fit(
    x_train, x_train,
    validation_data=(x_val, x_val),
    epochs=5,
    batch_size=128,
    shuffle=True,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# Plot training and validation loss
plt.figure(figsize=(7, 4))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Autoencoder Training vs Validation Loss')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# 5) Evaluate performance on test set
test_loss = autoencoder.evaluate(x_test, x_test, verbose=0)
print(f'Test MSE: {test_loss:.6f}')

In [ ]:
# 6) Original vs Reconstructed Images
n_images = 10
sample_images = x_test[:n_images]
reconstructions = autoencoder.predict(sample_images, verbose=0)

plt.figure(figsize=(18, 4))
for i in range(n_images):
    # Original
    ax = plt.subplot(2, n_images, i + 1)
    plt.imshow(sample_images[i].squeeze(), cmap='gray')
    plt.title('Original', fontsize=9)
    plt.axis('off')

    # Reconstructed
    ax = plt.subplot(2, n_images, i + 1 + n_images)
    plt.imshow(reconstructions[i].squeeze(), cmap='gray')
    plt.title('Reconstructed', fontsize=9)
    plt.axis('off')

plt.suptitle('Original vs Reconstructed Fashion-MNIST Images')
plt.tight_layout()
plt.show()

## 7) Reconstruction Quality
The images look very similar when reconstructed and, so the model did pretty well. The MSE between the image pixel values is also somewhat low being around than 0.007. The images blur from hte compression, but the general shape is still captured which is pretty good for scaling the 64 latent dimensons up to the  28x28 images.